In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("flights.csv")
df

,source,destination,cost,duration_minutes,flight_number,distance_miles
0,ATL,ORD,85,120,DL1234,606
1,ATL,DFW,125,150,DL2456,731
2,ATL,CLT,65,75,DL3789,227
3,ATL,MCO,75,105,DL4567,404
4,ATL,MIA,95,125,DL5678,595
...,...,...,...,...,...,...
472,PBI,MIA,58,55,AA2791,65
473,PBI,ATL,115,145,DL3802,548
474,PBI,MCO,72,75,B64913,130
475,PBI,CLT,135,165,AA5024,691


### Funtion to store graph as Adjacency List

In [3]:
def build_graph_from_df(df):
    #Initialise an empty dictionary
    # key: source node
    # value: list consisting of tuples. Example tuple: (destination, edge_data)
    graph = {} #dictionary based structure
    
    #iterate through each row of df
    for _, row in df.iterrows():
        source = row['source']
        destination = row['destination']
        
        #dictionary based structure to store edge data
        edge_data = {
            'cost': int(row['cost']),
            'duration_minutes': int(row['duration_minutes']),
            'flight_number': row['flight_number'],
            'distance_miles': int(row['distance_miles'])
        }
        
        #Check if key already exists. If not, add it to the graph and set value as empty list
        if source not in graph:
            graph[source] = []
            
        #Add the egde to the list for a certain "source" node
        graph[source].append((destination, edge_data))

    return graph


### Function to Print Graph

In [4]:
#Accepts graph dictionary created earlier and optional start airport
def print_graph(graph, start_airport=None):
    
    #Print info of flights only from the mentioned start airport
    if start_airport:
        print(f"\nFlights from {start_airport}:")
        for dest, info in graph.get(start_airport, []):
            print(f"{start_airport} to {dest} "
                  f"(Flight {info['flight_number']}, ${info['cost']}, "
                  f"{info['duration_minutes']} mins, {info['distance_miles']} miles)")
    else:
        #Print entire graph is start airport is not provided
        print("\nFull Flight Graph:")
        for src, edges in graph.items():
            for dest, info in edges:
                print(f"{src} to {dest} ({info['flight_number']})")


### Build graph using Flight Data

In [5]:
graph = build_graph_from_df(df)
# start_airport="ATL"
start_airport="PBI"
# end_airport="MCO"
# end_airport="ATL"
end_airport="FLL"
end_airport="MSP"
    

### Print all the edges for a single airport

In [6]:
print_graph(graph, start_airport)




Flights from PBI:
PBI to FLL (Flight B61680, $38, 35 mins, 43 miles)
PBI to MIA (Flight AA2791, $58, 55 mins, 65 miles)
PBI to ATL (Flight DL3802, $115, 145 mins, 548 miles)
PBI to MCO (Flight B64913, $72, 75 mins, 130 miles)
PBI to CLT (Flight AA5024, $135, 165 mins, 691 miles)
PBI to DFW (Flight AA6135, $185, 215 mins, 1109 miles)


### Print edges for all airports

In [11]:
# from queue import PriorityQueue
# max_flight = PriorityQueue()
# max_flight.put((-2,"Kishan"))
# max_flight.put((-1,"Kumar"))
# max_flight.put((-4,"S"))
# max_flight.put((-10,"A"))
# while max_flight:
#     print("Max",max_flight.get())
    

In [ ]:
# min_flight = PriorityQueue()
# min_flight.put((2,"Kishan"))
# min_flight.put((1,"Kumar"))
# min_flight.put((4,"S"))
# min_flight.put((10,"A"))
# while min_flight:
#     print("Min",min_flight.get())

# while min_flight:
#     print("Max",min_flight.get())

In [12]:
from queue import PriorityQueue
class PriorityQ():
    def __init__(self):
        self.min_heap = PriorityQueue()
        self.max_heap = PriorityQueue()
    def put(self,priority,data):
        self.min_heap.put((priority,data))
        self.max_heap.put((-priority,data))
    def get(self,type='min'):
        return self.min_heap.get()[1] if type.lower() == "min" else self.max_heap.get()[1]
    def empty(self,type='min'):
        return self.min_heap.empty() if type.lower() == "min" else self.max_heap.empty()
        

In [21]:
def getOptimalRoute(graph,start,end,factor='cost'):
    '''
    Description : Find optimal route between two airports using Dijkstra's algorithm.
    
    Parameters:
        graph (dict): Flight network as adjacency list
        start (str): Source airport code (e.g., 'SFO')
        end (str): Destination airport code (e.g., 'LAX')
        factor (str): Optimization criteria - 'cost' or 'duration_minutes'
    
    Return Type:
        tuple: (exploredCities, total_weight)
            - optimalRoute (list): Optimal route based on the algorithm
            - cost (float): Minimum cost or duration from start to end 
    '''
    pq = PriorityQ()
    pathCost = {}
    pq.put(0,start)
    pathCost[start] = 0
    maxWeight = float('inf')
    exploredCities = {}
    while not pq.empty():
        currentCity = pq.get()
        #Iterates the current city's neighbours node ie..next airports for the minumum cost
        for nextCity,data in graph.get(currentCity,[]):
            currentWeight = data.get(factor,0)
            # print(nextCity,currentWeight)
            # print(limit,currentCity)
            estimatedWeight = (pathCost.get(currentCity,maxWeight) + currentWeight)
            if pathCost.get(nextCity,maxWeight) > estimatedWeight:
                pathCost[nextCity] = estimatedWeight
                pq.put(estimatedWeight,nextCity)
                exploredCities[nextCity] = currentCity
    optimalRoute = []
    curr = end_airport
    while curr in exploredCities:
        optimalRoute.append(curr)
        curr = exploredCities[curr]
    optimalRoute.append(start_airport)
    print(f"Route : {optimalRoute[::-1]} and {factor} {pathCost[end_airport]}") 
    return (optimalRoute,pathCost[end_airport])

In [22]:
# start_airport="PBI"
# end_airport="MCO"
# end_airport="ATL"
# end_airport="FLL"
# end_airport="MSP"

# start_airport = "PBI"
# end_airport = "MSP"

# start_airport = "PBI"
# end_airport = "MCO"

# start_airport = "LAX"
# end_airport = "LAS"

# start_airport = "SFO"
# end_airport = "LAX"

# start_airport = "SEA"
# end_airport = "MIA"

start_airport = "PHX"
end_airport = "MIA"



factor = 'cost'
# factor = 'duration_minutes'

# Main graph data used
route,cost = getOptimalRoute(graph,start_airport,end_airport,factor)



Route : ['PHX', 'DFW', 'MIA'] and cost 244


In [23]:
# Dummy Graph Data to test if its is giving path and cost based on the start, end and the factor
start_airport = "A"
end_airport = "B"
factor = 'cost'
# factor = 'duration_minutes'
graphDummy = {
    "A": [
        ("B", {"cost": 100, "duration_minutes": 60}),   # Direct: expensive but fast
        ("C", {"cost": 30, "duration_minutes": 120})    # To hub: cheap but slow
    ],
    "B": [
        ("D", {"cost": 50, "duration_minutes": 40})
    ],
    "C": [
        ("B", {"cost": 20, "duration_minutes": 150}),   # From hub to B: cheap but very slow
        ("D", {"cost": 60, "duration_minutes": 50})
    ],
    "D": []
}
route,cost = getOptimalRoute(graphDummy,start_airport,end_airport,factor)


Route : ['A', 'C', 'B'] and cost 50
